In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import glob, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 22})
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from scipy import stats
import matplotlib.patches as mpatches
import pandas as pd

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
dset_ctrl = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_average_all_members_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_average_all_members_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

dset_ctrl_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_average_all_members_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_average_all_members_land_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

dset_ctrl_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_average_all_members_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
dset_sens_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_average_all_members_ocean_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)

ld = xr.DataArray(leads, dims='lead_time', name='lead_time')
dset_ctrl['lead_time']=ld
dset_sens['lead_time']=ld
dset_ctrl_land['lead_time']=ld
dset_sens_land['lead_time']=ld
dset_ctrl_ocean['lead_time']=ld
dset_sens_ocean['lead_time']=ld


em_ctrl = dset_ctrl.mean('member')
em_sens = dset_sens.mean('member')
em_ctrl_land = dset_ctrl_land.mean('member')
em_sens_land = dset_sens_land.mean('member')
em_ctrl_ocean = dset_ctrl_ocean.mean('member')
em_sens_ocean = dset_sens_ocean.mean('member')

# Convert the dataset to a DataFrame
dset_ctrl = dset_ctrl.to_dataframe().reset_index()
dset_sens = dset_sens.to_dataframe().reset_index()
dset_ctrl_land = dset_ctrl_land.to_dataframe().reset_index()
dset_sens_land = dset_sens_land.to_dataframe().reset_index()
dset_ctrl_ocean = dset_ctrl_ocean.to_dataframe().reset_index()
dset_sens_ocean = dset_sens_ocean.to_dataframe().reset_index()

In [ ]:
results = []

for lt in dset_ctrl['lead_time'].unique():
    x = dset_ctrl[dset_ctrl['lead_time'] == lt]['tas'].values
    y = dset_sens[dset_sens['lead_time'] == lt]['tas'].values
    
    stat, pval = stats.ttest_ind(x, y, equal_var=False)  # Welch
    
    results.append({
        'lead_time': lt,
        't_stat': stat,
        'p_value': pval
    })

res_df_global = pd.DataFrame(results)

In [ ]:
res_df_global

In [ ]:
results = []

for lt in dset_ctrl_land['lead_time'].unique():
    x = dset_ctrl_land[dset_ctrl_land['lead_time'] == lt]['tas'].values
    y = dset_sens_land[dset_sens_land['lead_time'] == lt]['tas'].values
    
    stat, pval = stats.ttest_ind(x, y, equal_var=False)  # Welch
    
    results.append({
        'lead_time': lt,
        't_stat': stat,
        'p_value': pval
    })

res_df_land = pd.DataFrame(results)

In [ ]:
res_df_land

In [ ]:
results = []

for lt in dset_ctrl_ocean['lead_time'].unique():
    x = dset_ctrl_ocean[dset_ctrl_ocean['lead_time'] == lt]['tas'].values
    y = dset_sens_ocean[dset_sens_ocean['lead_time'] == lt]['tas'].values
    
    stat, pval = stats.ttest_ind(x, y, equal_var=False)  # Welch
    
    results.append({
        'lead_time': lt,
        't_stat': stat,
        'p_value': pval
    })

res_df_ocean = pd.DataFrame(results)

In [ ]:
res_df_ocean

In [ ]:
fig,ax=plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', data=dset_sens, ax=ax, color='lightcoral', linecolor='r')
red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
sns.boxplot(x='lead_time', y='tas', data=dset_ctrl, ax=ax, color='lavender', linecolor='b')
blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
plt.legend(handles=[red_patch, blue_patch])

em_ctrl[var].plot(ax=ax,color='k', linestyle='--')
em_sens[var].plot(ax=ax,color='k')
ax.set_title('a) GLOBAL')

fig.savefig(f'{FIG_DIR}/{var}_bias_global_1999.png', dpi=600, bbox_inches = 'tight')

In [ ]:
fig,ax=plt.subplots(figsize=(12, 6))
sns.boxplot(x='lead_time', y='tas', data=dset_sens_land, ax=ax, color='lightcoral', linecolor='r')
red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
sns.boxplot(x='lead_time', y='tas', data=dset_ctrl_land, ax=ax, color='lavender', linecolor='b')
blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
plt.legend(handles=[red_patch, blue_patch])

em_ctrl_land[var].plot(ax=ax,color='k', linestyle='--')
em_sens_land[var].plot(ax=ax,color='k')
ax.set_title('b) LAND ONLY')

fig.savefig(f'{FIG_DIR}/{var}_bias_land_1999.png', dpi=600, bbox_inches = 'tight')

In [ ]:

dset_sens_ocean['exp'] = 'SENS'
dset_ctrl_ocean['exp'] = 'CTRL'

df = pd.concat([dset_sens_ocean, dset_ctrl_ocean])

fig,ax=plt.subplots(figsize=(12, 6))
#sns.boxplot(x='lead_time', y='tas', data=dset_sens_ocean, ax=ax, color='lightcoral', linecolor='r')
#red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
#sns.boxplot(x='lead_time', y='tas', data=dset_ctrl_ocean, ax=ax, color='lavender', linecolor='b')
#blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')

sns.boxplot(
    x='lead_time',
    y='tas',
    hue='exp',
    data=df,
    palette={'SENS': 'lightcoral', 'CTRL': 'lavender'},
    ax=ax
)

#plt.legend(handles=[red_patch, blue_patch])

em_ctrl_ocean[var].plot(ax=ax,color='k', linestyle='--')
em_sens_ocean[var].plot(ax=ax,color='k')
ax.set_title('c) OCEAN ONLY')

#fig.savefig(f'{FIG_DIR}/{var}_bias_ocean_1999.png', dpi=600, bbox_inches = 'tight')

In [ ]:
# Carica le immagini
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg
img1 = mpimg.imread(f'{FIG_DIR}/tas_bias_global_1999.png')
img2 = mpimg.imread(f'{FIG_DIR}/tas_bias_land_1999.png')
img3 = mpimg.imread(f'{FIG_DIR}/tas_bias_ocean_1999.png')

fig = plt.figure(figsize=(10, 8), constrained_layout=False)

# Definisci il layout: 1 righe, 3 colonne, con GridSpec
gs = GridSpec(1, 3, wspace=0.0, hspace=0.0)

# Aggiungi i sottoplot
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
images = [img1, img2, img3]

# Mostra immagini senza assi
for ax, img in zip(axes, images):
    ax.imshow(img)
    ax.axis('off')
# Applica layout compatto
plt.tight_layout(pad=0.0)
# Sistema il layout
#plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3_tas_global_bias_1999.png', dpi=600, bbox_inches = 'tight')